# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record set @ids available in the dataset
record_sets = metadata.record_sets
print(f"Available record sets (by @id): {len(record_sets)} found.")
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    if 'fields' in rs:
        print("  Fields (with @ids):")
        for field in rs['fields']:
            if isinstance(field, dict):
                print(f"   - Field @id: {field.get('@id', '?')}, label: {field.get('name', '?')}, dataType: {field.get('dataType', '?')}")
            else:
                print(f"   - Field @id: {field}")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather the list of record set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
# Load all identified record sets as DataFrames
dataframes = {}
for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            dataframes[rsid] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set @id: {rsid}")
        else:
            print(f"No records found for record set @id: {rsid}")
    except Exception as e:
        print(f"Error loading record set @id: {rsid}: {e}")

# Display columns for the main record set (pick the first if available)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in main record set (@id: {main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping by key attributes to prepare for analysis.

In [ ]:
# You may need to choose numeric and group fields based on the printed column names above.
import numpy as np

if dataframes:
    df = dataframes[main_record_set_id]

    # Try to select a numeric field for demonstration. Update these to match your dataset column names as needed.
    example_numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            example_numeric_field = col
            break

    if example_numeric_field:
        # Filter records by a chosen threshold
        threshold = np.nanmean(df[example_numeric_field])
        filtered_df = df[df[example_numeric_field] > threshold]
        print(f"Filtered records with '{example_numeric_field}' > mean threshold {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{example_numeric_field}_normalized"] = (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) / filtered_df[example_numeric_field].std()
        print(f"\nNormalized '{example_numeric_field}' for filtered records:")
        display(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head())

        # Try to group by a categorical field
        group_field = None
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and col != example_numeric_field:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found to group by.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and example_numeric_field:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[example_numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{example_numeric_field}'")
    plt.xlabel(example_numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Violin/box plot by group if available
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=example_numeric_field)
        plt.title(f"'{example_numeric_field}' by '{group_field}'")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*The FAIR² dataset provides a structured set of protein and peptide quantification records from mass spectrometry of human extracellular vesicles. Through the Croissant schema and `mlcroissant` tools, we:

- Inspected all record set definitions and field @ids for precise referencing.
- Dynamically loaded all available record sets as DataFrames for immediate analysis.
- Explored quantitative fields, performed normalization, and visualized their distributions.

You can adjust field selections and add custom analytics, referencing all dataset entities by their provided `@id` values. See the [dataset Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for more details on the data structure.